# AKOrN Training on MNIST - Configuración Máxima (GPU)

**Objetivo**: Entrenar AKOrN con configuración de máxima precisión en Google Colab GPU

**Método**: 🚀 Ejecutar desde VSCode usando extensión **vscode-colab**

**Parámetros**:
- `--epochs 200`
- `--n 4` (dimensión osciladores)
- `--L 3` (3 capas Kuramoto)
- `--T 5` (5 timesteps por capa)
- `--ch 128` (128 canales base)
- `--batchsize 128`

**Tiempo estimado**: ~18 horas en GPU T4

**Accuracy esperada**: ~99%

---

## 📝 Instrucciones VSCode-Colab

1. **Abrir en VSCode**: Ya estás aquí ✓
2. **Select Kernel**: Click arriba derecha → "Connect to Colab"
3. **Autenticar**: Permitir acceso a Google Colab (primera vez)
4. **Ejecutar celdas**: Click ▶️ en cada celda secuencialmente
5. **Monitorear**: Outputs aparecen directamente en VSCode

**IMPORTANTE**: 
- Mantén VSCode abierto durante las 18 horas de entrenamiento
- No suspendas tu Mac
- La conexión se mantiene automáticamente

## 1. Setup Inicial - Verificar GPU

In [1]:
import torch
import os

# Verificar GPU
print("🔍 Verificando GPU...")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA versión: {torch.version.cuda}")
    print(f"Memoria GPU total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU no disponible, usando CPU (muy lento)")

# Configurar para usar toda la GPU
torch.backends.cudnn.benchmark = True
print("\n✅ cuDNN benchmark habilitado")

🔍 Verificando GPU...
CUDA disponible: True
GPU: Tesla T4
CUDA versión: 12.6
Memoria GPU total: 15.83 GB

✅ cuDNN benchmark habilitado


## 2. Clonar Repositorio en Colab

**¿Por qué?** vscode-colab solo sincroniza el notebook, no los archivos del proyecto (train_classification.py, source/, etc.)

**Solución**: Clonar el repositorio de GitHub en el servidor de Colab

In [ ]:
# PASO CRÍTICO: Clonar repositorio para tener acceso a todo el código de AKOrN
# vscode-colab solo sincroniza este notebook, no train_classification.py ni source/

print("📦 Clonando repositorio desde GitHub...")
!git clone https://github.com/ACRCris/Proyecto-Inv.-teorica..git

print("\n📂 Navegando a carpeta de AKOrN...")
%cd Proyecto-Inv.-teorica./codigo/akorn

print("\n📍 Ubicación actual:")
!pwd

print("\n✅ Ahora Colab tiene acceso a:")
print("   - train_classification.py")
print("   - source/models/classification/knet.py")
print("   - source/evals/classification/adv_attacks.py")
print("   - Todos los módulos necesarios")

# Verificar que train_classification.py existe
import os
if os.path.exists('train_classification.py'):
    print("\n✓ train_classification.py encontrado")
else:
    print("\n⚠️ ERROR: train_classification.py no encontrado")

### 💡 Alternativas (si prefieres no clonar)

**Opción B - Google Drive**:
```python
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Proyecto-Inv.-teorica/codigo/akorn
```
*Requiere: Subir proyecto completo a Drive primero*

**Opción C - Subir archivos manualmente**:
```python
from google.colab import files
uploaded = files.upload()  # Arrastra archivos .py
```
*Tedioso si tienes muchos archivos*

**🎯 Recomendado**: Usar clonar repo (celda anterior) - Rápido y siempre actualizado

## 3. Instalar Dependencias

In [ ]:
# Instalar requirements de AKOrN
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q tensorboard tqdm einops ema-pytorch autoattack

print("✅ Dependencias instaladas")

# Verificar versiones
import torch
import torchvision
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")

## 4. Descargar MNIST

In [ ]:
import torchvision
from torchvision import transforms

# Crear directorio de datos
!mkdir -p ./data

# Descargar MNIST
print("📥 Descargando MNIST...")
transform = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
testset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

print(f"✅ Train: {len(trainset)} imágenes")
print(f"✅ Test: {len(testset)} imágenes")

## 5. Configurar Google Drive (Opcional - Para guardar checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Crear directorio para guardar resultados
RESULTS_DIR = '/content/drive/MyDrive/akorn_mnist_maxima'
!mkdir -p {RESULTS_DIR}
print(f"✅ Resultados se guardarán en: {RESULTS_DIR}")

## 6. Iniciar Entrenamiento - Configuración MÁXIMA

**ADVERTENCIA**: Este entrenamiento tomará ~18 horas. Asegúrate de:
1. Estar en Colab Pro/Pro+ para tiempo ilimitado de GPU
2. Tener Google Drive montado para no perder checkpoints
3. Habilitar guardado automático cada 25 epochs

In [ ]:
# Comando de entrenamiento completo
!python train_classification.py mnist_maxima_precision \
    --data mnist \
    --epochs 200 \
    --n 4 \
    --L 3 \
    --T 5 \
    --ch 128 \
    --batchsize 128 \
    --lr 0.0001 \
    --device cuda \
    --checkpoint_every 25 \
    --adveval_freq 10 \
    --pgdeval_freq 50

## 7. Monitoreo Durante Entrenamiento (En otra celda)

Ejecuta esto en paralelo para monitorear el progreso:

In [ ]:
# Ver logs en tiempo real
!tail -f runs/mnist_maxima_precision/events.out.tfevents.*

## 8. Lanzar TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/mnist_maxima_precision

# Alternativamente, usar ngrok para acceso externo:
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token("TU_TOKEN_AQUI")  # Obtén token en ngrok.com
# public_url = ngrok.connect(6006)
# print(f"TensorBoard disponible en: {public_url}")

## 9. Verificar Uso de GPU en Tiempo Real

In [ ]:
# Ejecutar cada 30 segundos para monitorear GPU
import time
import torch

while True:
    if torch.cuda.is_available():
        mem_allocated = torch.cuda.memory_allocated(0) / 1e9
        mem_reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"GPU Memoria: {mem_allocated:.2f} GB usada / {mem_reserved:.2f} GB reservada")
    time.sleep(30)
    
# Detener con: Runtime -> Interrupt execution

## 10. Guardar Checkpoints en Drive (Automático cada 25 epochs)

In [ ]:
# Copiar checkpoints a Drive automáticamente
import shutil
import glob

checkpoints = glob.glob('runs/mnist_maxima_precision/checkpoint_*.pth')
for ckpt in checkpoints:
    dest = f"{RESULTS_DIR}/{os.path.basename(ckpt)}"
    shutil.copy(ckpt, dest)
    print(f"✅ Copiado: {ckpt} → {dest}")

# Copiar también EMA
ema_files = glob.glob('runs/mnist_maxima_precision/ema_*.pth')
for ema in ema_files:
    dest = f"{RESULTS_DIR}/{os.path.basename(ema)}"
    shutil.copy(ema, dest)
    print(f"✅ Copiado EMA: {ema} → {dest}")

## 11. Cargar Checkpoint y Evaluar (Después del entrenamiento)

In [ ]:
import torch
from source.models.classification.knet import AKOrN

# Cargar modelo desde checkpoint
device = torch.device('cuda')

model = AKOrN(
    n=4,
    ch=128,
    out_classes=10,
    L=3,
    T=5,
    J="conv",
    ksizes=[9, 7, 5],
    ro_ksize=3,
    ro_N=2,
    norm="bn",
    c_norm="gn",
    gamma=1.0,
    use_omega=True,
    init_omg=1.0,
    global_omg=True,
    learn_omg=True,
    ensemble=1,
).to(device)

# Cargar pesos del checkpoint final
checkpoint = torch.load('runs/mnist_maxima_precision/checkpoint_epoch_200.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print("✅ Modelo cargado desde checkpoint epoch 200")

# Evaluar
model.eval()
correct = 0
total = 0

testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False)

with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\n🎯 Accuracy Final: {accuracy:.2f}%")

## 12. Análisis de Resultados

### Accuracy por Epoch (esperado):
- Epoch 10: ~87%
- Epoch 50: ~95%
- Epoch 100: ~97%
- Epoch 150: ~98%
- Epoch 200: ~99%

### Robustez Adversarial (esperado):
- Clean: ~99%
- Random Noise: ~96%
- FGSM (ε=8/255): ~90%
- PGD (ε=8/255): ~80%

## 13. Descargar Resultados Completos

In [ ]:
# Comprimir todos los resultados
!zip -r mnist_maxima_results.zip runs/mnist_maxima_precision/

# Descargar (click derecho en el archivo en el explorador de archivos)
from google.colab import files
files.download('mnist_maxima_results.zip')

print("✅ Archivo listo para descargar")

## 14. Notas Importantes

### Tiempo de Ejecución:
- GPU T4 (Colab Free): ~18-20 horas
- GPU V100 (Colab Pro+): ~10-12 horas
- GPU A100 (Colab Pro+): ~6-8 horas

### Memoria GPU Requerida:
- Batch size 128 con ch=128, L=3, T=5: ~8-10 GB
- Si encuentras OOM (Out of Memory):
  - Reducir `--batchsize 64`
  - O reducir `--ch 96`

### Guardado de Checkpoints:
- Cada 25 epochs (configurable con `--checkpoint_every`)
- Modelo base + EMA guardados por separado
- Total ~2 GB por checkpoint

### Reanudar Entrenamiento:
Si se interrumpe, añadir `--finetune runs/mnist_maxima_precision/checkpoint_epoch_XXX.pth`